# 02 VaR models: HS, EWMA and FHS

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from src.risk_measures import hs_var_es
from src.pipeline import build_models

m = build_models()
loss = m['loss']

## Historical simulation

In [ ]:
risk = hs_var_es(loss)
print(risk.dropna().head())
print(risk.dropna().describe())

In [ ]:
bt = risk.join(loss).dropna()

In [ ]:
hit = bt['loss'] > bt['var']
print(f'Days tested: {hit.index.size}')
print(f'Violations: {hit.sum()}')
print(f'Violation rate: {hit.mean()}')
print(f"Expected number of violations: {0.01 * hit.index.size}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(loss, label="Daily loss", color="grey", linewidth=0.6)
ax.plot(risk["var"], label="99% HS VaR (250-day window)", color="red", linewidth=1.2)
ax.scatter(bt.index[hit], bt['loss'][hit], color = 'orange', s = 12, zorder = 3, label = f'VaR Violations {hit.sum()}', edgecolors='black', linewidths=1)
ax.set_title("Historical-Simulation VaR vs Daily Loss, S&P 500")
ax.set_ylabel("Loss")
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.legend()

fig.tight_layout()
fig.savefig(ROOT / "results" / "hs_var.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from src.backtests import kupiec_test
kupiec_test(hit)

In [ ]:
from src.backtests import christoffersen_independence
christoffersen_independence(hit)

In [ ]:
from src.backtests import conditional_coverage
conditional_coverage(hit)

## EWMA (RiskMetrics)

In [ ]:
from src.risk_measures import ewma_var_es, ewma_volatility
ewma_risk = pd.DataFrame

sigma_ewma = ewma_volatility(losses=loss)
ewma_risk = ewma_var_es(sigma_ewma)

ewma_risk.tail()
bt_ewma = ewma_risk.join(loss).dropna()
hit_ewma = bt_ewma['loss'] > bt_ewma['VaR']

In [ ]:

fig, ax = plt.subplots()
ax.plot(loss, label="Daily loss", color="grey", linewidth=0.6)
ax.plot(ewma_risk["VaR"], label = r"99% EWMA VaR", color="teal", linewidth=1.2)
ax.scatter(bt_ewma.index[hit_ewma], bt_ewma['loss'][hit_ewma], color='coral', s=12, zorder=3,
           label=f'VaR Violations {hit_ewma.sum()}', edgecolors='black', linewidths=1)
ax.set_title("EWMA (RiskMetrics) VaR vs Daily Loss, S&P 500")
ax.set_ylabel("Loss")
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.legend()

fig.tight_layout()
fig.savefig(ROOT / "results" / "ewma_var.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
from src.backtests import backtest_summary
common = bt.index.intersection(bt_ewma.index)

hs_summary = backtest_summary(hit[common], 'Historical Simulation')
ewma_summary = backtest_summary(hit_ewma[common], 'EWMA')

compare = pd.concat([hs_summary, ewma_summary], axis=0)
compare


## Model comparison: HS, EWMA and FHS

In [ ]:
from src.risk_measures import fhs_var_es

fhs_risk = fhs_var_es(loss, sigma_ewma)
bt_fhs = fhs_risk.join(loss).dropna()
hit_fhs = bt_fhs['loss'] > bt_fhs['VaR']

common = bt.index.intersection(bt_ewma.index).intersection(bt_fhs.index)

compare = pd.concat([
    backtest_summary(hit[common], 'Historical Simulation'),
    backtest_summary(hit_ewma[common], 'EWMA'),
    backtest_summary(hit_fhs[common], 'FHS')])
compare

In [ ]:
start, end = "2020-01-15", "2020-07-01"

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(loss.loc[start:end], label="Daily loss", color="grey", linewidth=0.8)
ax.plot(risk["var"].loc[start:end], label="HS VaR", color="red", linewidth=1.2)
ax.plot(ewma_risk["VaR"].loc[start:end], label="EWMA VaR", color="teal", linewidth=1.2)
ax.plot(fhs_risk["VaR"].loc[start:end], label="FHS VaR", color="purple", linewidth=1.2)

ax.set_title("99% VaR around the COVID-19 crash, S&P 500")
ax.set_ylabel("Loss")
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.legend()
fig.tight_layout()
fig.savefig(ROOT / "results" / "var_comparison_2020.png", dpi=150, bbox_inches="tight")
plt.show()

## Robustness: decay factor and window length

In [ ]:
hits = {}
for lam in [0.94, 0.97, 0.99]:
    for w in [250, 500]:
        sig = ewma_volatility(loss, lam=lam)
        b = fhs_var_es(loss, sig, window=w).join(loss).dropna()
        hits[(lam, w)] = b["loss"] > b["VaR"]


common = None
for h in hits.values():
    common = h.index if common is None else common.intersection(h.index)

sens = pd.concat([backtest_summary(h[common], f"λ={lam}, w={w}")
                  for (lam, w), h in hits.items()])
sens